In [ ]:
import os
import json
import shutil

from pathlib import Path

import cv2
import numpy as np
import pandas as pd

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/yanhanr/lav-df-5000/LAV-DF-5000"

print("Dataset files:")
print(os.listdir(DATA_ROOT))

metadata_path = os.path.join(DATA_ROOT, "metadata.min.json")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("\nNumber of videos:", len(metadata))
print("\nFirst video:")
print(metadata[0])

In [ ]:
# --------------------------------------------------
# Helper: determine the 4 LAV-DF categories
# --------------------------------------------------

def get_category(item):
    mv = item["modify_video"]
    ma = item["modify_audio"]

    if not mv and not ma:
        return "real"
    elif not mv and ma:
        return "audio_only"
    elif mv and not ma:
        return "visual_only"
    else:
        return "both"


# --------------------------------------------------
# Helper: uniformly sample normal video timestamps
# --------------------------------------------------

def sample_normal_times(duration, n=3):
    # Avoid the exact first / last frame
    return np.linspace(
        duration / (n + 1),
        duration * n / (n + 1),
        n
    ).tolist()


# --------------------------------------------------
# Helper: sample timestamps INSIDE fake periods
# --------------------------------------------------

def sample_fake_times(fake_periods, duration, n=3):
    intervals = []

    for start, end in fake_periods:
        start = max(0.0, float(start))
        end = min(float(duration), float(end))

        if end > start:
            intervals.append((start, end))

    if not intervals:
        return []

    lengths = [end - start for start, end in intervals]
    total_fake_duration = sum(lengths)

    times = []

    # Pick n positions spread through all fake intervals
    for i in range(n):
        target = (i + 0.5) / n * total_fake_duration

        cumulative = 0.0

        for (start, end), length in zip(intervals, lengths):
            if target <= cumulative + length:
                t = start + (target - cumulative)
                times.append(t)
                break

            cumulative += length

    return times



## Preliminary Frame Dataset Preparation

This notebook prepares the initial frame dataset for preliminary visual deepfake detection experiments.

Up to three frames are extracted per video. Visually real frames are sampled uniformly across the video, while visually fake frames are sampled within annotated manipulation intervals.

The original LAV-DF TRAIN/DEV splits are preserved and saved as `train/val`. This lightweight dataset was used for initial model development before adopting the shared 10 FPS sampling protocol for standardized training and evaluation.

In [ ]:
FRAME_ROOT = Path("/kaggle/working/lavdf_frame")
FRAMES_PER_VIDEO = 3

# Delete old output if rerunning
if FRAME_ROOT.exists():
    shutil.rmtree(FRAME_ROOT)

# Folder structure for image classification
for split in ["train", "val"]:
    for label in ["real", "fake"]:
        (FRAME_ROOT / split / label).mkdir(
            parents=True,
            exist_ok=True
        )

records = []

# Preserve the official LAV-DF split:
# train -> train
# dev   -> val
split_map = {
    "train": "train",
    "dev": "val"
}

for lavdf_split, output_split in split_map.items():

    split_metadata = [
        item for item in metadata
        if item["split"] == lavdf_split
    ]

    print(
        f"\nProcessing {lavdf_split}: "
        f"{len(split_metadata)} videos"
    )

    for video_num, item in enumerate(
        split_metadata,
        start=1
    ):

        video_path = Path(DATA_ROOT) / item["file"]
        duration = float(item["duration"])
        category = get_category(item)

        # Visually manipulated video:
        # sample frames inside annotated fake periods
        if item["modify_video"]:

            times = sample_fake_times(
                item["fake_periods"],
                duration,
                FRAMES_PER_VIDEO
            )

            intended_label = 1

        # Visually unmodified video:
        # sample frames across the full video
        else:

            times = sample_normal_times(
                duration,
                FRAMES_PER_VIDEO
            )

            intended_label = 0

        cap = cv2.VideoCapture(
            str(video_path)
        )

        if not cap.isOpened():
            print(
                "Could not open:",
                video_path
            )
            continue

        fps = cap.get(
            cv2.CAP_PROP_FPS
        )

        if fps <= 0:
            cap.release()
            continue

        video_id = Path(
            item["file"]
        ).stem

        for i, target_time in enumerate(times):

            frame_idx = int(
                round(target_time * fps)
            )

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                frame_idx
            )

            success, frame = cap.read()

            if not success:
                continue

            actual_time = frame_idx / fps

            # Confirm whether the decoded frame
            # lies inside a visually manipulated interval
            inside_fake_period = any(
                start <= actual_time <= end
                for start, end
                in item["fake_periods"]
            )

            actual_label = int(
                item["modify_video"]
                and inside_fake_period
            )

            # Skip frames shifted across a fake-period boundary
            if actual_label != intended_label:
                continue

            label_name = (
                "fake"
                if actual_label
                else "real"
            )

            filename = (
                f"{lavdf_split}_"
                f"{video_id}_"
                f"{category}_"
                f"{i}.jpg"
            )

            output_path = (
                FRAME_ROOT
                / output_split
                / label_name
                / filename
            )

            cv2.imwrite(
                str(output_path),
                frame
            )

            records.append({
                "source_split": lavdf_split,
                "output_split": output_split,
                "video": item["file"],
                "video_id": video_id,
                "category": category,
                "timestamp": actual_time,
                "frame_index": frame_idx,
                "label": actual_label,
                "label_name": label_name,
                "frame_path": str(output_path)
            })

        cap.release()

        if video_num % 500 == 0:
            print(
                f"Processed "
                f"{video_num}/"
                f"{len(split_metadata)} videos"
            )


full_df = pd.DataFrame(records)

full_df.to_csv(
    FRAME_ROOT / "frames.csv",
    index=False
)

print("\nFinished!")

print("\nFrame counts:")

print(
    pd.crosstab(
        full_df["output_split"],
        full_df["label_name"]
    )
)

print(
    "\nTotal frames:",
    len(full_df)
)

print(
    "CSV:",
    FRAME_ROOT / "frames.csv"
)

In [ ]:
print(
    "lavdf_frame exists:",
    os.path.exists(
        "/kaggle/working/lavdf_frame"
    )
)

shutil.make_archive(
    "/kaggle/working/lavdf_frame",
    "zip",
    "/kaggle/working/lavdf_frame"
)

zip_path = (
    "/kaggle/working/"
    "lavdf_frame.zip"
)

print(
    "ZIP created:",
    os.path.exists(zip_path)
)

print(
    "ZIP size:",
    round(
        os.path.getsize(zip_path)
        / 1024**2,
        2
    ),
    "MB"
)